# P4 Agent 4 · Lexical Retrieval Index

| 항목 | 명세 |
|---|---|
| 목적 | Validate aliases and build the same-subcategory lexical index without dense reranking. |
| 담당 Agent | `P4-A4-NCS` |
| Stage ID | `A4-02-RETRIEVAL` |
| 입력 | `ncs_mapping/configs/ncs_alias_dictionary.yaml` |
| 처리 | alias 검증·lexical index·same-subcategory 제한 |
| 출력 | lexical index manifest 및 4개 종료 artifact |
| 선행 Gate | `NCS_CODESET_REVIEW_READY` |
| 후속 활용 | observed duty top-5 mapping |

> Development-only orchestration. Empirical analysis and production promotion are disabled.

In [ ]:
RUN_MODE = "observed-dev"
AGENT_ID = "P4-A4-NCS"
STAGE_ID = "A4-02-RETRIEVAL"
CONTRACT_VERSION = "2.1.2"
SCHEMA_VERSION = "ncs-retrieval-v1"
DATA_VERSION = "observed-dev-20260806.1"
CRAWL_RELEASE_ID = "CRAWL_20260806_03"
AS_OF_DATE = "2026-08-06"
INPUT_MANIFEST_PATH = "ncs_mapping/configs/ncs_alias_dictionary.yaml"
OUTPUT_ROOT = "ncs_mapping/data/runs/observed-dev/NCS_MAPPING_OBSERVED_20260806_01/A4-02-RETRIEVAL"
RANDOM_SEED = 20260806
FAIL_ON_GATE = True
EMPIRICAL_ANALYSIS_ALLOWED = False
DATA_PROVENANCE = "OBSERVED_DEVELOPMENT_ONLY"
PROMOTION_ALLOWED = False
DUTY_INPUT_PATH = ""
GOLD_INPUT_PATH = ""
CONTROL_SCHEMA_DIR = ""

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

NCS_ROOT = Path.cwd().resolve()
if NCS_ROOT.name != 'ncs_mapping':
    raise RuntimeError('run this notebook with cwd=ncs_mapping')
sys.path.insert(0, str(NCS_ROOT / 'src'))
assert RUN_MODE == 'observed-dev'
assert AGENT_ID == 'P4-A4-NCS' and STAGE_ID.startswith('A4-')
assert RANDOM_SEED == 20260806 and FAIL_ON_GATE is True
assert DATA_PROVENANCE == 'OBSERVED_DEVELOPMENT_ONLY'
assert EMPIRICAL_ANALYSIS_ALLOWED is False and PROMOTION_ALLOWED is False
resolved_duty_input = DUTY_INPUT_PATH or os.environ.get('P4_A2_DUTY_HANDOFF', '')
resolved_gold_input = GOLD_INPUT_PATH or os.environ.get('P4_NCS_GOLD_INPUT', '')
resolved_schema_dir = CONTROL_SCHEMA_DIR or os.environ.get('P4_CONTROL_SCHEMA_DIR', '')

In [ ]:
from p4_ncs.dictionary.alias_dictionary import load_alias_dictionary, validate_alias_dictionary
from p4_ncs.retrieval.lexical_index import LexicalIndex

ncs_units = pd.read_parquet(NCS_ROOT / 'data/processed/ncsUnit.parquet')
codeset = pd.read_parquet(NCS_ROOT / 'data/processed/coreAiItCodeSet.parquet')
aliases = load_alias_dictionary(NCS_ROOT / 'configs/ncs_alias_dictionary.yaml')
invalid_aliases = validate_alias_dictionary(aliases, set(codeset['ncsSubCode'].astype(str)))
lexical_index = LexicalIndex.build(ncs_units, codeset)
probe_hits = lexical_index.search('데이터 분석 모델 개발', top_k=5)
input_audit = {'aliases': len(aliases), 'invalidAliases': invalid_aliases, 'documents': len(lexical_index.documents), 'probeTopK': len(probe_hits), 'sameSubcategory': all(hit.matchedNcsUnitCode.startswith(hit.ncsSubCode) for hit in probe_hits)}
assert not invalid_aliases and input_audit['probeTopK'] <= 5 and input_audit['sameSubcategory']
input_audit

In [ ]:
from p4_ncs.workflow.observed import run_stage

stage_manifest = run_stage('A4-02-RETRIEVAL', root=NCS_ROOT, duty_input_path=resolved_duty_input or None, gold_input_path=resolved_gold_input or None, schema_dir=resolved_schema_dir or None)
stage_manifest

In [ ]:
stage_root = NCS_ROOT / 'data/runs' / RUN_MODE / 'NCS_MAPPING_OBSERVED_20260806_01' / stage_manifest['stageId']
expected_artifacts = {'stage_manifest.json', 'stage_metrics.json', 'stage_quality.csv', 'CHECKSUMS.sha256'}
actual_artifacts = {path.name for path in stage_root.iterdir() if path.is_file()}
assert actual_artifacts == expected_artifacts
termination_summary = {'stageId': stage_manifest['stageId'], 'status': stage_manifest['status'], 'rowCounts': stage_manifest['rowCounts'], 'artifacts': sorted(actual_artifacts)}
termination_summary